# Step 1: reproduce RAMP (official code) on Kaggle

Settings: **GPU T4 x2**, **Internet on**. Target: RAMP paper Table 24 (see `docs/ramp_reproduction.md`).

In [ ]:
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT
!pip install -q pyyaml
!bash scripts/ramp_official.sh setup

## Sanity check: the pretrained l_inf model (expect ~83.7 / 48.1 / 59.8 / 7.7 / 38.5)

In [ ]:
!bash scripts/ramp_official.sh pretr 0

## RAMP fine-tuning, 5 seeds, one queue per GPU

In [ ]:
!(bash scripts/ramp_official.sh train ramp "0 2 4" 0 & bash scripts/ramp_official.sh train ramp "1 3" 1 & wait)

In [ ]:
!(bash scripts/ramp_official.sh eval ramp "0 2 4" 0 & bash scripts/ramp_official.sh eval ramp "1 3" 1 & wait)
!python scripts/compare_targets.py --runs runs_official --map ramp_official_ft=ramp_l1.5 pretr_linf=pretr_linf
!grep -h "accuracy" external/ramp/trained_models/ramp_ft_s0/log_eval_final.txt  # official eval, seed 0, same 1000 points

## Optional: E-AT and MAX baselines (3 seeds each)

In [ ]:
!(bash scripts/ramp_official.sh train eat "0 1 2" 0 & bash scripts/ramp_official.sh train max "0 1 2" 1 & wait)
!(bash scripts/ramp_official.sh eval eat "0 1 2" 0 & bash scripts/ramp_official.sh eval max "0 1 2" 1 & wait)
!python scripts/compare_targets.py --runs runs_official --map ramp_official_ft=ramp_l1.5 eat_official_ft=eat max_official_ft=max pretr_linf=pretr_linf

In [ ]:
# keep the results: download this archive or save the notebook version
!tar czf /kaggle/working/ramp_results.tgz runs_official logs_official_*.txt